핵심 메시지: 방대한 데이터를 다룰 때, 이러한 알고리즘의 최적화는 처리 속도에 굉장히 중요

## DS2025_Hands-On3
### Solving the 0/1 knapsack optimization problem using different algorithms
We demonstrate a few classic approaches:

- **Greedy algorithms** for the Knapsack problem.
- **Exhaustive search** (search tree / backtracking) for the Knapsack problem.
- **Dynamic programming** via memoization for the Knapsack problem.

This code is adapted from **lecture2.py** of the MIT 6.0002 course material


## Importing Libraries

In [1]:
import numpy as np

## Defining the `Food` class
We’ll continue to use a `Food` class to represent individual items. Each item has:
- A **name** (string)
- A **value** (float or int)
- A **calorie** count (int)

We also define some helper methods:
- `getValue()` and `getCost()` to retrieve the respective attributes
- `density()` which returns `value / calorie`
- A `__str__` method for easy printing


In [2]:
class Food:
    def __init__(self, n, v, w):
        """
        n: Name of the food (string)
        v: Value of the food (int or float)
        w: Calorie cost of the food (int)
        """
        self.name = n
        self.value = v
        self.calories = w
    
    def getValue(self):
        return self.value

    def getCost(self):
        return self.calories

    def density(self):
        return self.getValue()/self.getCost()

    def __str__(self):
        return f"{self.name}: <{self.value}, {self.calories}>"

## Building a "Menu" of Foods with NumPy

The original function `buildMenu()` returned a Python list of `Food` objects. We’ll still build an array of `Food` objects, store them in a **NumPy array**. Each entry is one `Food` instance.


In [3]:
def buildMenu(names_array, values_array, calories_array):
    """
    names_array: np.array of strings
    values_array: np.array of numeric values
    calories_array: np.array of numeric values
    returns: np.array of Food objects
    """
    # Create a list of Food objects, then convert to np.array
    menu_list = []
    for i in range(len(names_array)):
        menu_list.append(Food(
            names_array[i],
            values_array[i],
            calories_array[i]
        ))
    return np.array(menu_list, dtype=object)  # store as an object array

## Greedy Approach
We sort the items by a given `keyFunction` in descending order and then select items until we run out of capacity.

### `greedy()`
1. Sort items by `keyFunction` (descending).
2. Iterate through the sorted items:
   - If adding the current item does not exceed the calorie limit, add it.
   - Keep track of total value and total calories used.


In [4]:
def greedy(items_array, maxCost, keyFunction):
    """
    items_array: np.array of Food objects
    maxCost: numeric, maximum calorie constraint
    keyFunction: function mapping Food -> numeric, used for sorting
    returns: (listOfTakenItems, totalValue)
    """
    # Convert items_array to a list so we can easily sort via Python's built-in sorted.
    items_list = list(items_array)
    # Sort descending by keyFunction
    itemsCopy = sorted(items_list, key=keyFunction, reverse=True)

    result = []
    totalValue, totalCost = 0.0, 0.0
    for item in itemsCopy:
        if (totalCost + item.getCost()) <= maxCost:
            result.append(item)
            totalCost += item.getCost()
            totalValue += item.getValue()

    return (result, totalValue)


### Testing the Greedy Algorithm
We’ll create a helper function to run a given greedy strategy, print the results, and see which items were chosen. Then `testGreedys()` tries three strategies:
1. Sort by **value** descending.
2. Sort by **cost** ascending (or 1/cost descending).
3. Sort by **density** (value/calorie) descending.

In [5]:
def testGreedy(items_array, constraint, keyFunction):
    taken, val = greedy(items_array, constraint, keyFunction)
    print('Total value of items taken =', val)
    for item in taken:
        print('   ', item)

def testGreedys(foods_array, maxUnits):
    print('Use greedy by value to allocate', maxUnits, 'calories')
    testGreedy(foods_array, maxUnits, Food.getValue)

    print('\nUse greedy by cost to allocate', maxUnits, 'calories')
    # Sorting by 1/cost is effectively sorting ascending by cost
    testGreedy(foods_array, maxUnits, lambda x: 1 / x.getCost())

    print('\nUse greedy by density to allocate', maxUnits, 'calories')
    testGreedy(foods_array, maxUnits, Food.density)


## Exhaustive Search (Recursive)
The function `maxVal()` performs a **0/1 Knapsack** via recursion:
1. If there are no items left or zero capacity, return 0.
2. If the first item’s cost exceeds our available calorie constraint, skip it.
3. Otherwise, recursively compute the best of:
   - Taking the item (if it fits)
   - Skipping the item

We compare which branch yields a larger total value.

In [6]:
def maxVal(toConsider, avail):
    """
    toConsider: list or array of Food objects
    avail: int, remaining calorie capacity
    returns: (maxValue, tupleOfItems)
    """
    # Because we frequently slice arrays, let's just treat it as a list for clarity.
    if len(toConsider) == 0 or avail == 0:
        result = (0, ())
    elif toConsider[0].getCost() > avail:
        # Explore skipping this item
        result =  maxVal(toConsider[1:], avail)
    else:
        nextItem = toConsider[0]
        # Explore taking nextItem
        withVal, withToTake = maxVal(toConsider[1:], avail - nextItem.getCost())
        withVal += nextItem.getValue()
        # Explore skipping nextItem
        withoutVal, withoutToTake = maxVal(toConsider[1:], avail)
        # Choose better option
        if withVal > withoutVal:
            result =  (withVal, withToTake + (nextItem,))
        else:
            result =  (withoutVal, withoutToTake)
    return result


### Testing the Exhaustive Search
We create a helper function `testMaxVal()` that calls the search function and prints the result.

In [7]:
def testMaxVal(foods_array, maxUnits, printItems=True):
    print('Use search tree to allocate', maxUnits, 'calories')
    # Convert to list for easy slicing
    foods_list = list(foods_array)
    val, taken = maxVal(foods_list, maxUnits)
    print('Total value of items taken =', val)
    if printItems:
        for item in taken:
            print('   ', item)

## Testing with a Small Example
We define a small set of items in **NumPy arrays**:
- `names`
- `values`
- `calories`

Then we build our menu using `buildMenu()`, and run both the **greedy** and **exhaustive** approaches.


In [8]:
# Example data
names = np.array(['wine', 'beer', 'pizza', 'burger', 'fries',
                 'cola', 'apple', 'donut'])
values = np.array([89, 90, 95, 100, 90, 79, 50, 10])
calories = np.array([123, 154, 258, 354, 365, 150, 95, 195])

# Build the menu of Food objects
foods = buildMenu(names, values, calories)

# Try a greedy approach
testGreedys(foods, 750)
print('')
# Try exhaustive search
testMaxVal(foods, 750)

Use greedy by value to allocate 750 calories
Total value of items taken = 284.0
    burger: <100, 354>
    pizza: <95, 258>
    wine: <89, 123>

Use greedy by cost to allocate 750 calories
Total value of items taken = 318.0
    apple: <50, 95>
    wine: <89, 123>
    cola: <79, 150>
    beer: <90, 154>
    donut: <10, 195>

Use greedy by density to allocate 750 calories
Total value of items taken = 318.0
    wine: <89, 123>
    beer: <90, 154>
    cola: <79, 150>
    apple: <50, 95>
    donut: <10, 195>

Use search tree to allocate 750 calories
Total value of items taken = 353
    cola: <79, 150>
    pizza: <95, 258>
    beer: <90, 154>
    wine: <89, 123>


## Building a Large Menu Randomly
We also provide a function `buildLargeMenu()` which creates random items. This is useful for testing efficiency of search vs. dynamic programming. We’ll store them in a NumPy array as well.


In [9]:
def buildLargeMenu(numItems, maxVal, maxCal):
    """
    numItems: number of items to generate
    maxVal: maximum possible value
    maxCal: maximum possible calorie cost
    returns: np.array of Food objects
    """
    name_list = []
    value_list = []
    cal_list = []

    for i in range(numItems):
        name_list.append(str(i))
        value_list.append(np.random.randint(1, maxVal))
        cal_list.append(np.random.randint(1, maxCal))

    # Convert to NumPy arrays
    names_array = np.array(name_list, dtype=object)
    values_array = np.array(value_list)
    cal_array = np.array(cal_list)

    return buildMenu(names_array, values_array, cal_array)

## Testing with a Larger Random Example
We build our menu using `buildLargeMenu()`, and run the **exhaustive** approach. What happens?


In [17]:
import time
numItemList = np.arange(5, 55, 5)
for numItems in numItemList:
    items = buildLargeMenu(numItems, 90, 250)
    start = time.time()
    testMaxVal(items, 750, True)
    end = time.time()
    print('Elapsed time: ',end-start)
    print('---')

Use search tree to allocate 750 calories
Total value of items taken = 215
    4: <34, 111>
    3: <83, 6>
    2: <40, 84>
    1: <17, 183>
    0: <41, 91>
Elapsed time:  0.00018978118896484375
---
Use search tree to allocate 750 calories
Total value of items taken = 242
    9: <43, 154>
    6: <56, 248>
    5: <55, 189>
    2: <67, 108>
    0: <21, 33>
Elapsed time:  0.0005309581756591797
---
Use search tree to allocate 750 calories
Total value of items taken = 488
    14: <58, 27>
    11: <59, 143>
    10: <83, 8>
    8: <64, 63>
    5: <75, 151>
    2: <66, 65>
    1: <7, 54>
    0: <76, 183>
Elapsed time:  0.010185003280639648
---
Use search tree to allocate 750 calories
Total value of items taken = 429
    17: <60, 8>
    13: <67, 7>
    8: <81, 240>
    5: <58, 183>
    4: <69, 108>
    1: <55, 51>
    0: <39, 99>
Elapsed time:  0.03516387939453125
---
Use search tree to allocate 750 calories
Total value of items taken = 674
    20: <63, 11>
    17: <57, 95>
    16: <65, 50>
    1

KeyboardInterrupt: 

## Dynamic Programming (Memoized) for Knapsack
Here is a memoized version of the **maxVal** function called `fastMaxVal()`. It avoids recomputing subproblems by caching results in a dictionary.

In [14]:
def fastMaxVal(toConsider, avail, memo=None):
    """
    toConsider: list or array of Food objects
    avail: int, calorie capacity
    memo: dict for memoization, keys = (len(toConsider), avail)
    returns: (maxValue, tupleOfItems)
    """
    if memo is None:
        memo = {}
    if (len(toConsider), avail) in memo:
        return memo[(len(toConsider), avail)]
    elif len(toConsider) == 0 or avail == 0:
        result = (0, ())
    elif toConsider[0].getCost() > avail:
        # Skip the first item
        result = fastMaxVal(toConsider[1:], avail, memo)
    else:
        nextItem = toConsider[0]
        # Explore taking item
        withVal, withToTake = fastMaxVal(toConsider[1:], avail - nextItem.getCost(), memo)
        withVal += nextItem.getValue()
        # Explore skipping item
        withoutVal, withoutToTake = fastMaxVal(toConsider[1:], avail, memo)
        # Compare
        if withVal > withoutVal:
            result = (withVal, withToTake + (nextItem,))
        else:
            result = (withoutVal, withoutToTake)

    memo[(len(toConsider), avail)] = result
    return result

### Testing `fastMaxVal`
We can reuse `testMaxVal`, but pass in the `fastMaxVal` function as our algorithm.

In [15]:
def testMaxVal_withAlgorithm(foods_array, maxUnits, algorithm, printItems=True):
    print('Menu contains', len(foods_array), 'items')
    print('Use search tree / DP to allocate', maxUnits, 'calories')
    val, taken = algorithm(list(foods_array), maxUnits)
    if printItems:
        print('Total value of items taken =', val)
        for item in taken:
            print('   ', item)

### Larger Random Menu using Dynamic Programming
Run the code below to see how larger random menus are handled.

In [16]:
import time
numItemList = np.arange(5, 55, 5)
for numItems in numItemList:
    items = buildLargeMenu(numItems, 90, 250)
    start = time.time()
    testMaxVal_withAlgorithm(items, 750, fastMaxVal, True)
    end = time.time()
    print('Elapsed time: ',end-start)
    print('---')

Menu contains 5 items
Use search tree / DP to allocate 750 calories
Total value of items taken = 279
    4: <29, 110>
    3: <64, 191>
    2: <73, 116>
    1: <31, 26>
    0: <82, 46>
Elapsed time:  0.00015687942504882812
---
Menu contains 10 items
Use search tree / DP to allocate 750 calories
Total value of items taken = 475
    9: <81, 187>
    8: <49, 39>
    5: <60, 102>
    4: <81, 63>
    3: <45, 83>
    2: <7, 45>
    1: <80, 31>
    0: <72, 195>
Elapsed time:  0.0010471343994140625
---
Menu contains 15 items
Use search tree / DP to allocate 750 calories
Total value of items taken = 631
    14: <72, 211>
    12: <61, 130>
    11: <30, 129>
    10: <76, 130>
    9: <70, 31>
    8: <34, 1>
    6: <35, 4>
    5: <2, 7>
    4: <51, 6>
    3: <76, 45>
    2: <83, 19>
    0: <41, 7>
Elapsed time:  0.006494998931884766
---
Menu contains 20 items
Use search tree / DP to allocate 750 calories
Total value of items taken = 489
    19: <81, 232>
    15: <83, 34>
    14: <64, 97>
    10: <65